# Baseline Model Training from Raw `.osz` (API, Max Opt)

This notebook trains only the baseline model using `train_api` (no CLI).
It keeps the same full-raw-data paths as `train_context_raw_data_api_maxopt.ipynb`, but uses the sample-style step-by-step structure and max-optimized data prep.


In [4]:
from pathlib import Path

import torch

from src.model import (
    ArchitectureSpec,
    TrainingSpec,
    WandbConfig,
    build_training_artifacts,
    create_training_context,
    load_training_context_from_checkpoint,
    prepare_sample_data_artifacts,
    train_context,
)

repo_root = Path.cwd()
raw_osz_dir = Path("E:/batchbeatmapdownloadtesttemp")
cache_root = Path("C:/taiko-transformer-cache")
data_root = cache_root# / "batchbeatmapdownloadtest"
training_dir = data_root / "training"
checkpoints_dir = repo_root / "checkpoints" / "baseline_maxopt"
last_checkpoint = checkpoints_dir / "last.ckpt"
best_checkpoint = checkpoints_dir / "best.ckpt"

index_cache_dir = training_dir / "index_cache"
inference_snapshots_dir = checkpoints_dir / "inference_snapshots"

epochs = 10
batch_size = 16
num_workers = 0
precision = "auto"
pin_memory = True
persistent_workers = False
prefetch_factor = 2
architecture_name = "taiko_transformer"
keep_only_max_notes_per_song = True
prepare_data = False
save_inference_every_n_steps = 1000
run_name = "baseline_raw_data_maxopt"
use_resume_if_available = True
use_wandb = False
wandb_log_every_batches = 100
wandb_notebook_name = "train_baseline_raw_data_api_maxopt.ipynb"
wandb_api_key = ""
wandb_offline = False

if torch.cuda.is_available():
    best_device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    best_device = "mps"
else:
    best_device = "cpu"

checkpoints_dir.mkdir(parents=True, exist_ok=True)

print(f"repo_root                     : {repo_root}")
print(f"raw_osz_dir                   : {raw_osz_dir}")
print(f"data_root                     : {data_root}")
print(f"training_dir                  : {training_dir}")
print(f"checkpoints_dir               : {checkpoints_dir}")
print(f"last_checkpoint               : {last_checkpoint}")
print(f"index_cache_dir   : {index_cache_dir}")
print(f"inference_snapshots_dir   : {inference_snapshots_dir}")
print(f"best_checkpoint               : {best_checkpoint}")
print(f"keep_only_max_notes_per_song  : {keep_only_max_notes_per_song}")
print(f"best_device                   : {best_device}")


repo_root                     : c:\Users\28548\PythonNotebooks\taiko-diffusion
raw_osz_dir                   : E:\batchbeatmapdownloadtesttemp
data_root                     : C:\taiko-transformer-cache
training_dir                  : C:\taiko-transformer-cache\training
checkpoints_dir               : c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_maxopt
last_checkpoint               : c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_maxopt\last.ckpt
index_cache_dir   : C:\taiko-transformer-cache\training\index_cache
inference_snapshots_dir   : c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_maxopt\inference_snapshots
best_checkpoint               : c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_maxopt\best.ckpt
keep_only_max_notes_per_song  : True
best_device                   : cuda


## Step 1: Prepare persistent dataset artifacts


In [5]:
if prepare_data:
    baseline_artifacts = prepare_sample_data_artifacts(
        osz_inputs=[str(raw_osz_dir)],
        data_root=data_root,
        keep_only_max_notes_per_song=keep_only_max_notes_per_song,
    )
else:
    baseline_artifacts = build_training_artifacts(data_root)
    print("Skipping raw-data preparation and reusing existing training artifacts.")

print(baseline_artifacts)


Skipping raw-data preparation and reusing existing training artifacts.
TrainingArtifacts(data_root=WindowsPath('C:/taiko-transformer-cache'), audio_dir=WindowsPath('C:/taiko-transformer-cache/beat_aligned_dataset/audio_npz'), token_dir=WindowsPath('C:/taiko-transformer-cache/beat_aligned_dataset/token_json'), chart_metadata_csv=WindowsPath('C:/taiko-transformer-cache/chart_index/chart_build_summary.csv'), sequence_metadata_csv=WindowsPath('C:/taiko-transformer-cache/beat_aligned_dataset/sequence_metadata.csv'), training_dir=WindowsPath('C:/taiko-transformer-cache/training'), splits_json=WindowsPath('C:/taiko-transformer-cache/training/splits.json'), vocab_json=WindowsPath('C:/taiko-transformer-cache/training/vocab.json'), checkpoints_dir=WindowsPath('C:/taiko-transformer-cache/training/checkpoints'))


## Step 2: Create or resume the baseline training context


In [6]:
baseline_architecture_spec = ArchitectureSpec(name=architecture_name)

baseline_training_spec = TrainingSpec(
    epochs=epochs,
    batch_size=batch_size,
    num_workers=num_workers,
    device=best_device,
    precision=precision,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor,
)

wandb_config = None
if use_wandb:
    wandb_config = WandbConfig(
        enabled=True,
        run_name=run_name,
        log_every_n_batches=wandb_log_every_batches,
        notebook_name=wandb_notebook_name,
        offline=wandb_offline,
        api_key=wandb_api_key,
        mode_name_for_run=architecture_name,
    )

if use_resume_if_available and last_checkpoint.exists():
    baseline_context = load_training_context_from_checkpoint(
        last_checkpoint,
        data_root=data_root,
        device=best_device,
        batch_size=batch_size,
        num_workers=num_workers,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print(f"Resuming from checkpoint: {last_checkpoint}")
else:
    baseline_context = create_training_context(
        data_root=data_root,
        architecture_spec=baseline_architecture_spec,
        training_spec=baseline_training_spec,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print("Starting a fresh baseline-model max-opt training run.")

print("baseline architecture:", baseline_context.architecture_spec)
print("baseline ignore_index:", baseline_context.dataset.label_ignore_index)
print("start_epoch:", baseline_context.start_epoch)
print("target_epochs:", epochs)


[startup] index cache lookup...
[startup] index cache lookup done in 0.00s
[startup] manifest...
[startup] manifest done in 1743.33s
[startup] splits...
[startup] splits done in 0.02s
[startup] indexes...
[startup] indexes done in 33.38s
[startup] index cache save...
[startup] index cache save done in 4.78s
[startup] vocab...
[startup] vocab done in 0.01s
[startup] dataset objects...
[startup] dataset objects done in 0.00s


c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting a fresh baseline-model max-opt training run.
baseline architecture: ArchitectureSpec(name='taiko_transformer', input_dim=128, d_model=256, nhead=4, num_encoder_layers=4, num_decoder_layers=4, dim_feedforward=1024, dropout=0.1, max_len=512, history_max_tokens=256, retrieval_top_k=1, retrieval_max_tokens_per_window=24, retrieval_exclude_last_n_windows=2, use_motif_retrieval=True, max_cached_charts=4)
baseline ignore_index: 0
start_epoch: 1
target_epochs: 10


## Step 3: Train the baseline model


In [7]:
baseline_context = train_context(
    baseline_context,
    epochs=epochs,
    log_every_n_batches=wandb_log_every_batches,
    wandb_config=wandb_config,
    save_inference_every_n_steps=save_inference_every_n_steps,
    inference_snapshots_dir=inference_snapshots_dir,
)

print("Training finished.")
print(f"last checkpoint: {last_checkpoint.resolve()}")
print(f"best checkpoint: {best_checkpoint.resolve()}")
print(f"inference snapshots dir: {inference_snapshots_dir.resolve()}")


[runtime] precision requested=auto resolved=bf16 autocast=1 scaler=0
[runtime] inference_snapshots every_n_steps=1000 dir=C:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_maxopt\inference_snapshots


Training:   0%|          | 0/139866 [00:00<?, ?it/s]c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


KeyboardInterrupt: 

## Optional inspection


In [ ]:
print("history keys:", baseline_context.history.keys())
print("training dir:", training_dir)
print("vocab json:", training_dir / "vocab.json")
print("splits json:", training_dir / "splits.json")
